In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.geometry import Point
import matplotlib.pyplot as plt
import os

### 20 ~ 23년도 유동인구 데이터 전처리 후 데이터 프레임
#### 성연령별 : pop_gender_age
#### 시간별   : pop_time
#### 요일별   : pop_day

In [ ]:
#국토부 주민등록인구 100M 격자 데이터
sihueng_grid100 = gpd.read_file('../P1_Data/01.External/국토부/국토부_시흥시_100m격자_총인구수/(B100)국토통계_인구정보-총 인구 수(전체)-(격자) 100M_경기도 시흥시_202304/nlsp_021001001.shp', encoding='utf-8')
sihueng_grid100

### 시흥시 생활인구 성연령별,요일별,시간별 100m 격자 데이터

In [ ]:
# 성연령별 2020 ~ 2023
merged_sex_age = gpd.read_file('../P1_Data/02.Pretreatment/merged_sex_age(20~23).shp',encoding='cp949')
# 시간별 2020 ~ 2023
merged_time = gpd.read_file('../P1_Data/02.Pretreatment/merged_time(20~23).shp',encoding='cp949')
# 요일별 2020 ~ 2023
merged_day = gpd.read_file('../P1_Data/02.Pretreatment/merged_day(20~23).shp',encoding='cp949')
#복사
pop_gender_age = merged_sex_age.copy()
pop_time = merged_time.copy()
pop_day = merged_day.copy()

In [ ]:
# '날짜컬럼'을 문자열 컬럼으로 변경
pop_gender_age['etl_ym'] = pop_gender_age['etl_ym'].astype(str)

# 연도 컬럼 생성 (앞 4자리)
pop_gender_age['Year'] = pop_gender_age['etl_ym'].str[:4]
# 월 컬럼 생성 (뒤 2자리)
pop_gender_age['Month'] = pop_gender_age['etl_ym'].str[4:]
pop_gender_age

### 행정경계 데이터

In [ ]:
## 전국 센서스 행정경계 데이터
sensus = gpd.read_file('../P1_Data/01.External/국토부/Z_SOP_BND_ADM_DONG_PG/Z_SOP_BND_ADM_DONG_PG.shp',encoding='cp949')
## 시흥시만 추출 후 시흥시 경계면 저장
siheung = sensus[sensus['ADM_CD'].str.startswith('3115')]
siheung.to_file('../P1_Data/02.Pretreatment/siheung_boundary.shp', encoding = 'cp949')
#시흥시 행정경계 불러오기
siheung = gpd.read_file('../P1_Data/02.Pretreatment/siheung_boundary.shp', encoding = 'cp949')
siheung

In [ ]:
#거북섬동 추가된 행정경계데이터 _ 0926 업데이트
siheung = gpd.read_file('../P1_Data/01.External/행정동경계 (거북섬)/siheung_boundary(202308).shp',encoding='utf-8')
siheung

### 시흥시 경계 및 읍면동 정보와 시흥시 생활인구 100m 격자 매핑

In [ ]:
siheung_LP = pop_gender_age.copy()

siheung_LP['living_pop'] = siheung_LP['M00_09'] + siheung_LP['M10_19'] + siheung_LP['M20_29'] + siheung_LP['M30_39'] + siheung_LP['M40_49'] + siheung_LP['M50_59'] + siheung_LP['M60_up'] + siheung_LP['F00_09'] + siheung_LP['F10_19'] + siheung_LP['F20_29'] + siheung_LP['F30_39'] + siheung_LP['F40_49'] + siheung_LP['F50_59'] + siheung_LP['F60_up']

siheung_LP = siheung_LP[['GRID_100M_','Year','Month','living_pop','geometry']]
siheung_LP

In [ ]:
siheung_LP = pop_gender_age.copy()

siheung_LP['living_pop'] = siheung_LP['M00_09'] + siheung_LP['M10_19'] + siheung_LP['M20_29'] + siheung_LP['M30_39'] + siheung_LP['M40_49'] + siheung_LP['M50_59'] + siheung_LP['M60_up'] + siheung_LP['F00_09'] + siheung_LP['F10_19'] + siheung_LP['F20_29'] + siheung_LP['F30_39'] + siheung_LP['F40_49'] + siheung_LP['F50_59'] + siheung_LP['F60_up']

siheung_LP = siheung_LP[['GRID_100M_','Year','Month','living_pop','geometry']]
siheung_LP

In [ ]:
siheung_LP_2020 = siheung_LP[siheung_LP['Year'] == '2020']
siheung_LP_2021 = siheung_LP[siheung_LP['Year'] == '2021']
siheung_LP_2022 = siheung_LP[siheung_LP['Year'] == '2022']
siheung_LP_2023 = siheung_LP[siheung_LP['Year'] == '2023']

In [ ]:
##2020년도 처리
# gid로 그룹화하고 people 컬럼 값을 더함
siheung_LP_2020 = siheung_LP_2020.groupby(['GRID_100M_','Year','geometry'])['living_pop'].sum().reset_index()
siheung_LP_2020 = siheung_LP_2020[['GRID_100M_','Year','living_pop','geometry']]
siheung_LP_2020 = gpd.GeoDataFrame(siheung_LP_2020, geometry='geometry', crs='EPSG:5179')

#좌표계 통일
siheung = siheung.to_crs(siheung_LP_2020.crs)

mapping = gpd.sjoin(siheung_LP_2020, siheung,how='left')

# 특정 컬럼 값이 중복되는 행 추출
duplicate_rows = mapping[mapping.duplicated(subset='GRID_100M_', keep=False)]
#mapping한 기존격자에서 중복 되는 행 제거
mapping = mapping.drop_duplicates(subset='GRID_100M_', keep=False)


##중복되는 행의 처리를 위한 작업
#중복되는 행들과 시흥 경계면 오버레이하여 걸쳐진 부분 면적 구함
res_union = gpd.overlay(duplicate_rows, siheung, how='intersection')

# 두 컬럼의 값이 동일한 행 추출
same_res_union = res_union[res_union['ADM_NM_1'] == res_union['ADM_NM_2']]

#면적계산
same_res_union['area'] = same_res_union['geometry'].area

# 같은 GID 값끼리 그룹화하여 면적컬럼에서 가장 큰 값을 가진 읍면동컬럼으로 업데이트
result_union = same_res_union.groupby('GRID_100M_').apply(lambda x: x.loc[x['area'].idxmax()]).reset_index(drop=True)
# 행정동 매핑한 격자와 여러개의 행정동이 걸처진 격자처리 후 두개의 데이터프레임을 다시 합치는 과정
mapping = mapping[['GRID_100M_','ADM_CD','ADM_NM','living_pop','geometry']]

result_union = result_union[['GRID_100M_', 'ADM_CD_1', 'ADM_NM_1', 'living_pop', 'geometry']]
# 'A' 컬럼의 이름을 'X'로 변경
result_union.rename(columns={'ADM_CD_1': 'ADM_CD'}, inplace=True)
result_union.rename(columns={'ADM_NM_1': 'ADM_NM'}, inplace=True)


#결과
siheung_grid_2020 = pd.concat([mapping, result_union], axis=0)
# 인덱스를 리셋합니다.
siheung_grid_2020 = siheung_grid_2020.reset_index(drop=True)
siheung_grid_2020

In [ ]:
##2021년도 처리
# gid로 그룹화하고 people 컬럼 값을 더함
siheung_LP_2021 = siheung_LP_2021.groupby(['GRID_100M_','Year','geometry'])['living_pop'].sum().reset_index()
siheung_LP_2021 = siheung_LP_2021[['GRID_100M_','Year','living_pop','geometry']]
siheung_LP_2021 = gpd.GeoDataFrame(siheung_LP_2021, geometry='geometry', crs='EPSG:5179')

#좌표계 통일
siheung = siheung.to_crs(siheung_LP_2021.crs)

mapping = gpd.sjoin(siheung_LP_2021, siheung,how='left')

# 특정 컬럼 값이 중복되는 행 추출
duplicate_rows = mapping[mapping.duplicated(subset='GRID_100M_', keep=False)]
#mapping한 기존격자에서 중복 되는 행 제거
mapping = mapping.drop_duplicates(subset='GRID_100M_', keep=False)


##중복되는 행의 처리를 위한 작업

#중복되는 행들과 시흥 경계면 오버레이하여 걸쳐진 부분 면적 구함
res_union = gpd.overlay(duplicate_rows, siheung, how='intersection')

# 두 컬럼의 값이 동일한 행 추출
same_res_union = res_union[res_union['ADM_NM_1'] == res_union['ADM_NM_2']]

#면적계산
same_res_union['area'] = same_res_union['geometry'].area

# 같은 GID 값끼리 그룹화하여 면적컬럼에서 가장 큰 값을 가진 읍면동컬럼으로 업데이트
result_union = same_res_union.groupby('GRID_100M_').apply(lambda x: x.loc[x['area'].idxmax()]).reset_index(drop=True)
# 행정동 매핑한 격자와 여러개의 행정동이 걸처진 격자처리 후 두개의 데이터프레임을 다시 합치는 과정
mapping = mapping[['GRID_100M_','ADM_CD','ADM_NM','living_pop','geometry']]


result_union = result_union[['GRID_100M_','ADM_CD_1','ADM_NM_1','living_pop','geometry']]
# 'A' 컬럼의 이름을 'X'로 변경
result_union.rename(columns={'ADM_CD_1': 'ADM_CD'}, inplace=True)
result_union.rename(columns={'ADM_NM_1': 'ADM_NM'}, inplace=True)


#결과
siheung_grid_2021 = pd.concat([mapping, result_union], axis=0)
# 인덱스를 리셋합니다.
siheung_grid_2021 = siheung_grid_2021.reset_index(drop=True)
siheung_grid_2021

In [ ]:
##2022년도 처리
# gid로 그룹화하고 people 컬럼 값을 더함
siheung_LP_2022 = siheung_LP_2022.groupby(['GRID_100M_','Year','geometry'])['living_pop'].sum().reset_index()
siheung_LP_2022 = siheung_LP_2022[['GRID_100M_','Year','living_pop','geometry']]
siheung_LP_2022 = gpd.GeoDataFrame(siheung_LP_2022, geometry='geometry', crs='EPSG:5179')

#좌표계 통일
siheung = siheung.to_crs(siheung_LP_2022.crs)

mapping = gpd.sjoin(siheung_LP_2022, siheung,how='left')

# 특정 컬럼 값이 중복되는 행 추출
duplicate_rows = mapping[mapping.duplicated(subset='GRID_100M_', keep=False)]
#mapping한 기존격자에서 중복 되는 행 제거
mapping = mapping.drop_duplicates(subset='GRID_100M_', keep=False)


##중복되는 행의 처리를 위한 작업

#중복되는 행들과 시흥 경계면 오버레이하여 걸쳐진 부분 면적 구함
res_union = gpd.overlay(duplicate_rows, siheung, how='intersection')

# 두 컬럼의 값이 동일한 행 추출
same_res_union = res_union[res_union['ADM_NM_1'] == res_union['ADM_NM_2']]

#면적계산
same_res_union['area'] = same_res_union['geometry'].area

# 같은 GID 값끼리 그룹화하여 면적컬럼에서 가장 큰 값을 가진 읍면동컬럼으로 업데이트
result_union = same_res_union.groupby('GRID_100M_').apply(lambda x: x.loc[x['area'].idxmax()]).reset_index(drop=True)
# 행정동 매핑한 격자와 여러개의 행정동이 걸처진 격자처리 후 두개의 데이터프레임을 다시 합치는 과정
mapping = mapping[['GRID_100M_','ADM_CD','ADM_NM','living_pop','geometry']]


result_union = result_union[['GRID_100M_','ADM_CD_1','ADM_NM_1','living_pop','geometry']]
# 'A' 컬럼의 이름을 'X'로 변경
result_union.rename(columns={'ADM_CD_1': 'ADM_CD'}, inplace=True)
result_union.rename(columns={'ADM_NM_1': 'ADM_NM'}, inplace=True)


#결과
siheung_grid_2022 = pd.concat([mapping, result_union], axis=0)
# 인덱스를 리셋합니다.
siheung_grid_2022 = siheung_grid_2022.reset_index(drop=True)
siheung_grid_2022

In [ ]:
##2023년도 처리
# gid로 그룹화하고 people 컬럼 값을 더함
siheung_LP_2023 = siheung_LP_2023.groupby(['GRID_100M_','Year','geometry'])['living_pop'].sum().reset_index()
siheung_LP_2023 = siheung_LP_2023[['GRID_100M_','Year','living_pop','geometry']]
siheung_LP_2023 = gpd.GeoDataFrame(siheung_LP_2023, geometry='geometry', crs='EPSG:5179')

#좌표계 통일
siheung = siheung.to_crs(siheung_LP_2023.crs)

mapping = gpd.sjoin(siheung_LP_2023, siheung,how='left')

# 특정 컬럼 값이 중복되는 행 추출
duplicate_rows = mapping[mapping.duplicated(subset='GRID_100M_', keep=False)]
#mapping한 기존격자에서 중복 되는 행 제거
mapping = mapping.drop_duplicates(subset='GRID_100M_', keep=False)


##중복되는 행의 처리를 위한 작업

#중복되는 행들과 시흥 경계면 오버레이하여 걸쳐진 부분 면적 구함
res_union = gpd.overlay(duplicate_rows, siheung, how='intersection')

# 두 컬럼의 값이 동일한 행 추출
same_res_union = res_union[res_union['ADM_NM_1'] == res_union['ADM_NM_2']]

#면적계산
same_res_union['area'] = same_res_union['geometry'].area

# 같은 GID 값끼리 그룹화하여 면적컬럼에서 가장 큰 값을 가진 읍면동컬럼으로 업데이트
result_union = same_res_union.groupby('GRID_100M_').apply(lambda x: x.loc[x['area'].idxmax()]).reset_index(drop=True)
# 행정동 매핑한 격자와 여러개의 행정동이 걸처진 격자처리 후 두개의 데이터프레임을 다시 합치는 과정
mapping = mapping[['GRID_100M_','ADM_CD','ADM_NM','living_pop','geometry']]


result_union = result_union[['GRID_100M_','ADM_CD_1','ADM_NM_1','living_pop','geometry']]
# 'A' 컬럼의 이름을 'X'로 변경
result_union.rename(columns={'ADM_CD_1': 'ADM_CD'}, inplace=True)
result_union.rename(columns={'ADM_NM_1': 'ADM_NM'}, inplace=True)


#결과
siheung_grid_2023 = pd.concat([mapping, result_union], axis=0)
# 인덱스를 리셋합니다.
siheung_grid_2023 = siheung_grid_2023.reset_index(drop=True)
siheung_grid_2023

In [ ]:
siheung_grid_2020.to_file('../P3_Result/03.SHP/시흥시_100M격자_생활인구(행정동)_2020년도.shp', encoding = 'cp949')
siheung_grid_2021.to_file('../P3_Result/03.SHP/시흥시_100M격자_생활인구(행정동)_2021년도.shp', encoding = 'cp949')
siheung_grid_2022.to_file('../P3_Result/03.SHP/시흥시_100M격자_생활인구(행정동)_2022년도.shp', encoding = 'cp949')
siheung_grid_2023.to_file('../P3_Result/03.SHP/시흥시_100M격자_생활인구(행정동)_2023년도.shp', encoding = 'cp949')